In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
BATCH_INFO_PATH = "data/raw/batch_info_orien_all.csv"
CLINICAL_PATH   = "data/processed/clinical_orien_aligned_v4.csv"

In [3]:
batch = pd.read_csv(BATCH_INFO_PATH, index_col=0)
clin  = pd.read_csv(CLINICAL_PATH, index_col=0)

In [4]:
batch

,ORIENAvatarKey,SLID,NexusBatch,MappingRate,UniqueRateofMapped,DuplicateRateofMapped,BaseMismatch,ExpressionProfilingEfficiency,ExonicRate,IntronicRate,...,Median3Primebias,MedianofAvgTranscriptCoverage,MedianofTranscriptCoverageStd,pf_reads,q30,DV200,CaptureKit,Sequencer_Type,Isolation_Method,QCCheck
1,E8CR36NB5W,AZSFH08235-RNA,30-911421228-RNA,0.936603,0.304574,0.695426,0.005294,0.713197,0.763434,0.033072,...,0.333333,2.993600,4.60118,136273754,93.470001,78.9,TruSeq,NovaSeq X,Custom,Pass
2,5B2GXZDY45,AZSFH08238-RNA,30-911421228-RNA,0.896320,0.308808,0.691192,0.005039,0.692627,0.776494,0.040845,...,0.333333,0.926641,1.76922,98094112,93.250000,83.6,TruSeq,NovaSeq X,Custom,Pass
3,5B2GXZDY45,AZSFH08239-RNA,30-911421228-RNA,0.916191,0.358694,0.641306,0.005724,0.672890,0.735997,0.043523,...,0.336735,2.539630,3.87381,85695712,92.480003,75.9,TruSeq,NovaSeq X,Custom,Pass
4,A5D0NDUHUO,AZSFH08240-RNA,30-911421228-RNA,0.925779,0.377563,0.622437,0.004514,0.695233,0.753201,0.054269,...,0.321553,3.096690,4.34506,115232660,93.180000,68.4,TruSeq,NovaSeq X,Custom,Pass
5,UP3W0UO3YR,AZSFH08246-RNA,30-911421228-RNA,0.913631,0.322658,0.677342,0.004440,0.702603,0.771519,0.041349,...,0.333333,5.572880,7.39128,150380136,93.680000,62.2,TruSeq,NovaSeq X,Custom,Pass
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1383,GTJSQY0H9W,SL619881,RNA Batch 228,0.876975,0.077360,0.922640,0.004846,0.688876,0.787754,0.055912,...,0.363636,1.709890,4.75773,144592670,93.120000,24.9,TruSeq,NovaSeq,Covaris/KF/DP_RNA,Flag
1384,NUGRETY77D,SL619884,RNA Batch 228,0.913564,0.072998,0.927002,0.004913,0.717897,0.787820,0.054905,...,0.333333,2.910340,7.63337,173020203,95.090000,41.1,TruSeq,NovaSeq,Covaris/KF/DP_RNA,Flag
1385,FRS145H0TA,SL660645,RNA Batch 243,0.865664,0.119505,0.880495,0.005479,0.662855,0.768283,0.064543,...,0.348100,1.952080,4.74699,125679788,95.820000,52.5,TruSeq,NovaSeq,Covaris/KF/DP_RNA,Pass
1386,E6XY0R6H2V,SL660686,RNA Batch 243,0.886152,0.104027,0.895973,0.004552,0.686434,0.776869,0.064855,...,0.333333,1.587440,4.05077,150076546,96.610000,68.1,TruSeq,NovaSeq,Covaris/KF/DP_RNA,Pass


In [5]:
KEY = "ORIENAvatarKey"

In [6]:
if "sampleID" in clin.columns:
    clin_keys = clin["sampleID"].astype(str)
elif clin.index.name in ("sampleID", KEY):
    clin_keys = clin.index.astype(str)
else:
    raise ValueError(f"Cannot find patient key. Columns: {list(clin.columns)}; index: {clin.index.name}")


In [7]:
batch[KEY] = batch[KEY].astype(str)

print(f"batch_info rows            : {len(batch)}")
print(f"batch_info unique {KEY}: {batch[KEY].nunique()}")
print(f"clinical rows              : {len(clin)}")
print(f"clinical unique keys       : {clin_keys.nunique()}")

batch_info rows            : 1387
batch_info unique ORIENAvatarKey: 1286
clinical rows              : 1112
clinical unique keys       : 1112


In [8]:
dup_mask = batch[KEY].duplicated(keep=False)
dups = batch.loc[dup_mask].sort_values(KEY)

n_dup_keys = dups[KEY].nunique()
print(f"Keys appearing >1 time : {n_dup_keys}")
print(f"Rows involved          : {len(dups)}")

if n_dup_keys == 0:
    print("\nNo duplicates. match() first-row behaviour is harmless. Q1 closed.")
else:
    g = dups.groupby(KEY)
    summary = pd.DataFrame({
        "n_rows":        g.size(),
        "n_batches":     g["NexusBatch"].nunique(),
        "dv200_min":     g["DV200"].min(),
        "dv200_max":     g["DV200"].max(),
        "n_isolation":   g["Isolation_Method"].nunique(),
        "n_sequencer":   g["Sequencer_Type"].nunique(),
    })
    straddling = summary[summary["n_batches"] > 1]

    print(f"\nKeys straddling >1 NexusBatch : {len(straddling)}")
    print(f"Keys within a single batch     : {(summary['n_batches'] == 1).sum()}")
    print(f"Keys with >1 isolation method  : {(summary['n_isolation'] > 1).sum()}")
    print(f"Max within-key DV200 spread    : {(summary['dv200_max'] - summary['dv200_min']).max():.1f}")

    cols = [KEY, "SLID", "NexusBatch", "Isolation_Method", "DV200",
            "UniqueRateofMapped", "Sequencer_Type", "ReadLength", "QCCheck"]
    display(dups[[c for c in cols if c in dups.columns]])

Keys appearing >1 time : 99
Rows involved          : 200

Keys straddling >1 NexusBatch : 17
Keys within a single batch     : 82
Keys with >1 isolation method  : 6
Max within-key DV200 spread    : 46.7


,ORIENAvatarKey,SLID,NexusBatch,Isolation_Method,DV200,UniqueRateofMapped,Sequencer_Type,ReadLength,QCCheck
978,0J4GJRS3EA,SL433582,RNA Batch 176,Covaris/KF/DP_RNA,48.1,0.328182,NovaSeq,101,Pass
967,0J4GJRS3EA,SL432495,RNA Batch 175,Covaris/KF/DP_RNA,73.3,0.325817,NovaSeq,101,Pass
817,0YZGPRFFGI,SL398281,RNA Batch 110,Covaris/KF/DP_RNA,55.4,0.307629,NovaSeq,76,Pass
865,0YZGPRFFGI,SL414894,RNA Batch 137,Covaris/KF/DP_RNA,79.1,0.395241,NovaSeq,76,Pass
1073,1UR7IQQRS8,SL558654,RNA Batch 217,Covaris/KF/DP_RNA,70.0,0.241433,NovaSeq,101,Pass
...,...,...,...,...,...,...,...,...,...
893,ZIWWE1DFCW,SL421126,RNA Batch 154,Covaris/KF/DP_RNA,62.8,0.256221,NovaSeq,101,Pass
755,ZO84U069AF,SL371166,RNA Batch 77,Covaris/KF_RNA,77.4,0.205380,NovaSeq,76,Pass
756,ZO84U069AF,SL371188,RNA Batch 77,Covaris/KF_RNA,78.9,0.223527,NovaSeq,76,Pass
528,ZXA1FK4M2K,FT-SA274774R,FT-SPN05758,Covaris/KF_RNA,76.3,0.409634,NovaSeqX,151,Pass


In [9]:
batch_keys = set(batch[KEY])
clin_set   = set(clin_keys)

missing = clin_set - batch_keys
extra   = batch_keys - clin_set

print(f"Analysable patients             : {len(clin_set)}")
print(f"  present in batch_info         : {len(clin_set & batch_keys)}")
print(f"  MISSING from batch_info       : {len(missing)}")
print(f"batch_info keys not analysable  : {len(extra)}")

if missing:
    print("\nMissing keys (first 20):", sorted(missing)[:20])

Analysable patients             : 1112
  present in batch_info         : 1112
  MISSING from batch_info       : 0
batch_info keys not analysable  : 174


In [10]:
sub = batch[batch[KEY].isin(clin_set)].drop_duplicates(subset=KEY, keep="first")
print(f"Rows retained (one per analysable patient): {len(sub)}\n")

print("--- Isolation_Method ---")
print(sub["Isolation_Method"].value_counts(dropna=False).to_string())

ffpe = sub["Isolation_Method"].str.contains("FFPE", na=False)
print(f"\nFFPE-labelled : {ffpe.sum()}")
print(f"Non-FFPE      : {(~ffpe).sum()}")

print("\n--- DV200 ---")
print(sub["DV200"].describe().round(1).to_string())
print(f"\nDV200 < 30 : {(sub['DV200'] < 30).sum()}")
print(f"DV200 == 0 : {(sub['DV200'] == 0).sum()}")
print(f"DV200 NaN  : {sub['DV200'].isna().sum()}")

print("\n--- DV200 by FFPE status ---")
print(sub.groupby(ffpe)["DV200"].describe().round(1).to_string())

print("\n--- QCCheck ---")
print(sub["QCCheck"].value_counts(dropna=False).to_string())

# FFPE nesting within batch
bt = sub.groupby("NexusBatch").agg(n=(KEY, "size"), ffpe_frac=("Isolation_Method",
        lambda s: s.str.contains("FFPE", na=False).mean()))
print(f"\nBatches (analysable subset) : {len(bt)}")
print(f"  all-FFPE       : {(bt.ffpe_frac == 1).sum()}")
print(f"  all non-FFPE   : {(bt.ffpe_frac == 0).sum()}")
print(f"  mixed          : {((bt.ffpe_frac > 0) & (bt.ffpe_frac < 1)).sum()}")

print(f"\nBatch sizes: singletons={ (bt.n == 1).sum() }, "
      f"size<3={ (bt.n < 3).sum() } ({ bt.n[bt.n < 3].sum() } samples)")
print(f"FFPE samples in batches of size<3: "
      f"{ ffpe[sub['NexusBatch'].isin(bt.index[bt.n < 3])].sum() }")

Rows retained (one per analysable patient): 1112

--- Isolation_Method ---
Isolation_Method
Covaris/KF_RNA                 488
Covaris/KF/DP_RNA              285
RNeasy Plus Mini               240
AllPrep FFPE                    48
Covaris FFPE Extraction Kit     32
Custom                           8
AllPrep DNA/RNA mini Kit         6
Tissue_HC_200_v7                 3
Preisolated material             1
No_Protocol                      1

FFPE-labelled : 80
Non-FFPE      : 1032

--- DV200 ---
count    1112.0
mean       69.9
std        20.7
min         0.0
25%        58.7
50%        74.2
75%        85.4
max        98.0

DV200 < 30 : 53
DV200 == 0 : 4
DV200 NaN  : 0

--- DV200 by FFPE status ---
                   count  mean   std   min   25%   50%   75%   max
Isolation_Method                                                  
False             1032.0  72.3  17.4  19.2  61.0  75.0  86.6  98.0
True                80.0  40.0  33.0   0.0   7.2  33.4  75.4  92.2

--- QCCheck ---
QCCheck
Pass

In [32]:
OUTDIR = "results/rung1_meta_heterogeneity"
from pathlib import Path
OUTDIR = Path("results/rung1_meta_heterogeneity")
PG     = OUTDIR / "per_gene_results.csv"

CHI2_1_MEDIAN = 0.4549364
LAMBDA_REPORTED = 1.415701604761061      # from global_summary.json, FULL

df = pd.read_csv(PG, index_col=0)
n_before = len(df)
df = df[~df.index.duplicated(keep="first")]
if len(df) != n_before:
    print(f"dropped {n_before - len(df)} duplicated-index rows from the symbol join")

b1 = df["beta_tcga"].values;  s1 = df["se_tcga"].values
b2 = df["beta_orien"].values; s2 = df["se_orien"].values
N  = len(df)
print(f"genes: {N}")
# ---------------------------------------------------------------- moment estimators
mse1, mse2 = float(np.mean(s1**2)), float(np.mean(s2**2))
Sxx, Syy   = float(np.var(b1, ddof=1)), float(np.var(b2, ddof=1))
Sxy        = float(np.cov(b1, b2, ddof=1)[0, 1])
V1, V2     = Sxx - mse1, Syy - mse2

c_reg  = Sxy / V1                      # regression slope of true b2 on true b1
tau2   = V2 - c_reg**2 * V1            # equation error: divergence not explained by scale
r_true = Sxy / np.sqrt(V1 * V2)

# Deming: delta = var(err in y) / var(err in x); assumes tau2 == 0
delta   = mse2 / mse1
disc    = (Syy - delta*Sxx)**2 + 4*delta*Sxy**2
c_deming = ((Syy - delta*Sxx) + np.sqrt(disc)) / (2*Sxy)

# contribution to the Q excess:  E[Q] - 1  ~  [(1-c)^2 V1 + tau2] / (mse1 + mse2)
scale_term = (1 - c_reg)**2 * V1
print("\n--- calibration ---")
print(f"  V_hat  T / O            : {V1:.6f} / {V2:.6f}   (ratio {V2/V1:.2f})")
print(f"  mean_se2 T / O          : {mse1:.6f} / {mse2:.6f}")
print(f"  Cov(true)               : {Sxy:.6f}")
print(f"  c_reg (Cov/V1)          : {c_reg:.4f}")
print(f"  c_deming (delta={delta:.3f}) : {c_deming:.4f}")
print(f"  r_true                  : {r_true:.4f}   (r_true^2 = {r_true**2:.4f} = shared frac of V2)")
print(f"  tau^2 (equation error)  : {tau2:.6f}")
print(f"  scale term (1-c)^2 V1   : {scale_term:.8f}")
print(f"  tau2 : scale term       : {tau2/scale_term:.1f} : 1")
print(f"  scale share of excess   : {100*scale_term/(scale_term+tau2):.2f}%")

# ---------------------------------------------------------------- lambda grid
def lam_and_tails(a):
    Q = (b1 - b2/a)**2 / (s1**2 + (s2/a)**2)
    return (float(np.median(Q)/CHI2_1_MEDIAN),
            float(np.mean(Q > stats.chi2.ppf(0.95, 1))),
            float(np.mean(Q > stats.chi2.ppf(0.99, 1))),
            float(np.mean(Q > stats.chi2.ppf(0.999, 1))))

lam1 = lam_and_tails(1.0)[0]
assert abs(lam1 - LAMBDA_REPORTED) < 1e-9, f"a=1 gives {lam1}, expected {LAMBDA_REPORTED}"
print(f"\nself-consistency at a=1: lambda={lam1:.6f}  OK")

grid = np.round(np.concatenate([np.arange(0.50, 4.001, 0.02), [c_reg, c_deming]]), 4)
grid = np.unique(grid)
rows = [dict(zip(["a","lambda","tail95","tail99","tail999"], (a,)+lam_and_tails(a))) for a in grid]
g = pd.DataFrame(rows)
g.to_csv(OUTDIR /  "calibration_grid.csv", index=False)

amin = g.loc[g["lambda"].idxmin()]
lam_creg = float(g.loc[np.isclose(g.a, round(c_reg,4)), "lambda"].iloc[0])
lam_dem  = float(g.loc[np.isclose(g.a, round(c_deming,4)), "lambda"].iloc[0])

# asymptote: as a -> inf, Q -> z1^2, i.e. TCGA's own marginal inflation
lam_tcga_only = float(np.median((b1/s1)**2)/CHI2_1_MEDIAN)

print("\n--- lambda under global rescaling of ORIEN ---")
print(f"  a = 1        (as fitted)  : {lam1:.4f}")
print(f"  a = {c_reg:.3f}   (c_reg)      : {lam_creg:.4f}")
print(f"  a = {c_deming:.3f}   (c_deming)   : {lam_dem:.4f}")
print(f"  grid minimum at a={amin.a:.2f}   : {amin['lambda']:.4f}   <-- FLOOR (bound, not an estimate)")
print(f"  a -> inf  (TCGA alone)    : {lam_tcga_only:.4f}")
print(f"\n  tails at floor 95/99/99.9 : {amin.tail95:.4f}/{amin.tail99:.4f}/{amin.tail999:.4f}"
      f"   (expected 0.05/0.01/0.001)")

# ---------------------------------------------------------------- top-k overlap
z1, z2 = np.abs(b1/s1), np.abs(b2/s2)
genes = df.index.values
topk = {}
print("\n--- top-k overlap by |z| ---")
print(f"  {'k':>5} {'obs':>5} {'exp':>7} {'fold':>6} {'p_hyper':>10}")
for k in (50, 100, 500, 1000):
    st = set(genes[np.argsort(-z1)[:k]]); so = set(genes[np.argsort(-z2)[:k]])
    ov  = len(st & so); exp = k*k/N
    p   = float(stats.hypergeom.sf(ov-1, N, k, k))
    topk[k] = {"observed": ov, "expected": exp, "fold": ov/exp if exp else np.nan, "p_hyper": p}
    print(f"  {k:>5} {ov:>5} {exp:>7.1f} {ov/exp:>6.2f} {p:>10.2e}")

# ---------------------------------------------------------------- save
with open(OUTDIR /  "calibration_summary.json", "w") as f:
    json.dump({
        "n_genes": N,
        "V_hat_tcga": V1, "V_hat_orien": V2, "mean_se2_tcga": mse1, "mean_se2_orien": mse2,
        "cov_true": Sxy, "c_reg": c_reg, "c_deming": c_deming, "delta": delta,
        "r_true": r_true, "tau2": tau2, "scale_term": scale_term,
        "scale_share_of_excess": scale_term/(scale_term+tau2),
        "lambda_a1": lam1, "lambda_at_c_reg": lam_creg, "lambda_at_c_deming": lam_dem,
        "lambda_floor": float(amin["lambda"]), "a_at_floor": float(amin.a),
        "lambda_tcga_only_asymptote": lam_tcga_only,
        "topk_overlap": topk,
    }, f, indent=2)
print(f"\nwritten: {OUTDIR / 'calibration_grid.csv'}, {OUTDIR /  'calibration_summary.json'}")

dropped 14 duplicated-index rows from the symbol join
genes: 14675

--- calibration ---
  V_hat  T / O            : 0.001623 / 0.004849   (ratio 2.99)
  mean_se2 T / O          : 0.007036 / 0.002267
  Cov(true)               : 0.001890
  c_reg (Cov/V1)          : 1.1645
  c_deming (delta=0.322) : 2.4219
  r_true                  : 0.6737   (r_true^2 = 0.4539 = shared frac of V2)
  tau^2 (equation error)  : 0.002648
  scale term (1-c)^2 V1   : 0.00004390
  tau2 : scale term       : 60.3 : 1
  scale share of excess   : 1.63%

self-consistency at a=1: lambda=1.415702  OK

--- lambda under global rescaling of ORIEN ---
  a = 1        (as fitted)  : 1.4157
  a = 1.164   (c_reg)      : 1.3523
  a = 2.422   (c_deming)   : 1.2227
  grid minimum at a=2.36   : 1.2212   <-- FLOOR (bound, not an estimate)
  a -> inf  (TCGA alone)    : 1.3070

  tails at floor 95/99/99.9 : 0.0708/0.0147/0.0017   (expected 0.05/0.01/0.001)

--- top-k overlap by |z| ---
      k   obs     exp   fold    p_hyper
     50

In [24]:
print("rows:", len(df), " unique index:", df.index.nunique(),
      " duplicated:", int(df.index.duplicated().sum()))

rows: 14689  unique index: 14675  duplicated: 14


In [26]:
dups = df.index[df.index.duplicated(keep=False)] if "gene_symbol" not in df else None
d = pd.read_csv(PG, index_col=0)
print(d.loc[d.index.duplicated(keep=False), ["gene_symbol"]].sort_index().to_string())

                gene_symbol
gene_id                    
ENSG00000002586        CD99
ENSG00000002586        CD99
ENSG00000124333       VAMP7
ENSG00000124333       VAMP7
ENSG00000167393     PPP2R3B
ENSG00000167393     PPP2R3B
ENSG00000169084       DHRSX
ENSG00000169084       DHRSX
ENSG00000169093       ASMTL
ENSG00000169093       ASMTL
ENSG00000169100     SLC25A6
ENSG00000169100     SLC25A6
ENSG00000178605      GTPBP6
ENSG00000178605      GTPBP6
ENSG00000182162       P2RY8
ENSG00000182162       P2RY8
ENSG00000182378      PLCXD1
ENSG00000182378      PLCXD1
ENSG00000182484      WASH6P
ENSG00000182484      WASH6P
ENSG00000185291       IL3RA
ENSG00000185291       IL3RA
ENSG00000197976     AKAP17A
ENSG00000197976     AKAP17A
ENSG00000198223      CSF2RA
ENSG00000198223      CSF2RA
ENSG00000214717       ZBED1
ENSG00000214717       ZBED1


In [35]:
from pathlib import Path
from sklearn.utils.extmath import randomized_svd

PROC = Path("data/processed")
N_PCS, B_PERM, SEED = 300, 200, 42
KS = [1, 5, 10, 25, 50, 100, 200, 300]
rng = np.random.default_rng(SEED)

def _load(coh):
    expr = pd.read_csv(PROC / f"{coh}_preprocessed_v4.csv", index_col=0)
    surv = pd.read_csv(PROC / f"surv_{coh}_aligned_v4.csv")
    skey = "sampleID" if "sampleID" in surv.columns else surv.columns[0]
    surv = surv.set_index(skey)[["time", "event"]]
    missing = [g for g in genes if g not in expr.index]
    assert not missing, f"{coh}: {len(missing)} genes absent"
    expr = expr.loc[genes]                       # enforce b1/b2 gene order
    smp  = [s for s in expr.columns if s in surv.index]
    expr, surv = expr[smp], surv.loc[smp]
    return (expr.values.T.astype(np.float64),
            surv["time"].values.astype(np.float64),
            surv["event"].values.astype(np.float64))

Xt, tt, et = _load("tcga")
Xo, to, eo = _load("orien")
    
Xt = Xt - Xt.mean(0)
Xo = Xo - Xo.mean(0)
print("centred:", np.abs(Xt.mean(0)).max(), np.abs(Xo.mean(0)).max())
print(f"TCGA {Xt.shape}  ORIEN {Xo.shape}  genes={N}")
assert np.abs(Xt.mean(0)).max() < 0.05 and np.abs(Xo.mean(0)).max() < 0.05, "columns not centred"

centred: 6.519694649771476e-16 1.0364411167836107e-15
TCGA (339, 14675)  ORIEN (1112, 14675)  genes=14675


In [36]:
def cox_one_step(X, time, event):
    """Score-test estimates at beta=0, all columns. Breslow ties. Returns (U/V, 1/sqrt(V))."""
    n, p = X.shape
    o = np.argsort(time, kind="mergesort")
    Xs, ts, es = X[o], time[o], event[o]
    r1 = np.zeros(p); r2 = np.zeros(p); U = np.zeros(p); V = np.zeros(p)
    i = n - 1
    while i >= 0:
        j = i
        while j > 0 and ts[j-1] == ts[i]:
            j -= 1
        blk = Xs[j:i+1]
        r1 += blk.sum(0); r2 += (blk**2).sum(0)
        nR = n - j
        ev = es[j:i+1] == 1
        ne = int(ev.sum())
        if ne:
            xbar = r1 / nR
            U += blk[ev].sum(0) - ne * xbar
            V += ne * (r2 / nR - xbar**2)
        i = j - 1
    V = np.maximum(V, 1e-12)
    return U / V, 1.0 / np.sqrt(V)

b1_os, s1_os = cox_one_step(Xt, tt, et)
b2_os, s2_os = cox_one_step(Xo, to, eo)
print("--- one-step vs MLE ---")
for nm, a, b in [("beta tcga", b1, b1_os), ("se tcga", s1, s1_os),
                 ("beta orien", b2, b2_os), ("se orien", s2, s2_os)]:
    print(f"  {nm:11s} r={np.corrcoef(a,b)[0,1]:.6f}  median ratio={np.median(b/a):.4f}")

--- one-step vs MLE ---
  beta tcga   r=0.997488  median ratio=1.0052
  se tcga     r=0.789970  median ratio=1.0033
  beta orien  r=0.999148  median ratio=0.9995
  se orien    r=0.876933  median ratio=0.9994


/tmp/ipykernel_2170/3813922003.py:30: RuntimeWarning: divide by zero encountered in divide
  print(f"  {nm:11s} r={np.corrcoef(a,b)[0,1]:.6f}  median ratio={np.median(b/a):.4f}")


In [37]:
def gene_pcs(X, tag):
    _, S, Vt = randomized_svd(X, n_components=N_PCS, random_state=SEED)
    ev = S**2 / (S**2).sum()
    print(f"  {tag}: PC1={ev[0]*100:.1f}%  top10={ev[:10].sum()*100:.1f}%  "
          f"top{N_PCS}={ev.sum()*100:.1f}%")
    return Vt

print("--- gene-space PCs ---")
PCS = {"tcga": gene_pcs(Xt, "TCGA"), "orien": gene_pcs(Xo, "ORIEN")}
cum_frac = lambda d, Vt: np.cumsum((Vt @ d)**2) / float(d @ d)

--- gene-space PCs ---
  TCGA: PC1=4.8%  top10=24.0%  top300=100.0%
  ORIEN: PC1=25.6%  top10=60.8%  top300=100.0%


In [39]:
d_mle  = b1 - b2
ds_mle = d_mle / np.sqrt(s1**2 + s2**2)
d_os   = b1_os - b2_os
ds_os  = d_os / np.sqrt(s1_os**2 + s2_os**2)

obs = {pc: {"raw_mle": cum_frac(d_mle, Vt), "std_mle": cum_frac(ds_mle, Vt),
            "raw_os":  cum_frac(d_os,  Vt), "std_os":  cum_frac(ds_os,  Vt)}
       for pc, Vt in PCS.items()}

null = {pc: {"raw": [], "std": []} for pc in PCS}
print(f"--- permutation null (B={B_PERM}) ---")
for b in range(B_PERM):
    it, io = rng.permutation(len(tt)), rng.permutation(len(to))
    pb1, ps1 = cox_one_step(Xt, tt[it], et[it])
    pb2, ps2 = cox_one_step(Xo, to[io], eo[io])
    d = pb1 - pb2
    ds = d / np.sqrt(ps1**2 + ps2**2)
    for pc, Vt in PCS.items():
        null[pc]["raw"].append(cum_frac(d, Vt))
        null[pc]["std"].append(cum_frac(ds, Vt))
    if (b+1) % 25 == 0: print(f"  {b+1}/{B_PERM}")

out = {}
for pc in PCS:
    for scale in ("raw", "std"):
        nl = np.array(null[pc][scale]); ob = obs[pc][f"{scale}_os"]; obm = obs[pc][f"{scale}_mle"]
        print(f"\n=== {pc.upper()} PCs | {scale} d ===")
        print(f"  {'K':>4} {'obs(MLE)':>9} {'obs(1step)':>11} {'null mean':>10} {'null SD':>8} {'z':>7} {'emp p':>8}")
        rows = []
        for k in KS:
            nk = nl[:, k-1]
            z  = (ob[k-1] - nk.mean()) / nk.std(ddof=1)
            pv = (1 + (nk >= ob[k-1]).sum()) / (len(nk) + 1)
            print(f"  {k:>4} {obm[k-1]:>9.4f} {ob[k-1]:>11.4f} {nk.mean():>10.4f} "
                  f"{nk.std(ddof=1):>8.4f} {z:>7.2f} {pv:>8.4f}")
            rows.append({"K": k, "obs_mle": float(obm[k-1]), "obs_onestep": float(ob[k-1]),
                         "null_mean": float(nk.mean()), "null_sd": float(nk.std(ddof=1)),
                         "z": float(z), "emp_p": float(pv)})
        out[f"{pc}_{scale}"] = rows

with open(OUTDIR / "projection_check.json", "w") as f:
    json.dump({"n_genes": int(N), "n_pcs": N_PCS, "B_perm": B_PERM, "results": out}, f, indent=2)
print(f"\nwritten: {OUTDIR / 'projection_check.json'}")

--- permutation null (B=200) ---
  25/200
  50/200
  75/200
  100/200
  125/200
  150/200
  175/200
  200/200

=== TCGA PCs | raw d ===
     K  obs(MLE)  obs(1step)  null mean  null SD       z    emp p
     1    0.1325      0.1430     0.0554   0.0651    1.34   0.1045
     5    0.1857      0.1959     0.1580   0.0842    0.45   0.3234
    10    0.2586      0.2686     0.2249   0.0891    0.49   0.2935
    25    0.4229      0.4314     0.3416   0.0790    1.14   0.1393
    50    0.4865      0.4948     0.4357   0.0702    0.84   0.1940
   100    0.5452      0.5538     0.5439   0.0591    0.17   0.3930
   200    0.6620      0.6699     0.7002   0.0388   -0.78   0.8109
   300    0.7380      0.7453     0.8192   0.0246   -3.00   0.9900

=== TCGA PCs | std d ===
     K  obs(MLE)  obs(1step)  null mean  null SD       z    emp p
     1    0.1378      0.1409     0.0555   0.0650    1.31   0.1095
     5    0.1932      0.1966     0.1592   0.0844    0.44   0.3284
    10    0.2661      0.2696     0.2268   0.08

In [40]:
for f in ("expr_tmp_tcga.csv", "expr_tmp_orien_1112.csv"):
    e = pd.read_csv(Path("data/raw") / f, index_col=0, nrows=200)
    full_cols = pd.read_csv(Path("data/raw") / f, index_col=0, nrows=0).shape[1]
    v = e.values
    print(f"\n{f}")
    print(f"  first 200 genes x {full_cols} samples")
    print(f"  index[:2]   : {list(e.index[:2])}")
    print(f"  columns[:2] : {list(e.columns[:2])}")
    print(f"  range       : [{np.nanmin(v):.3f}, {np.nanmax(v):.3f}]  median={np.nanmedian(v):.3f}")
    print(f"  zeros       : {100*np.mean(v==0):.1f}%")
    print(f"  in common_genes_v4: {sum(g in set(genes) for g in e.index)}/200")


expr_tmp_tcga.csv
  first 200 genes x 339 samples
  index[:2]   : ['ENSG00000000003', 'ENSG00000000005']
  columns[:2] : ['TCGA.CV.7423', 'TCGA.CR.7377']
  range       : [0.000, 18793.300]  median=9.365
  zeros       : 3.5%
  in common_genes_v4: 181/200

expr_tmp_orien_1112.csv
  first 200 genes x 1112 samples
  index[:2]   : ['ENSG00000000003', 'ENSG00000000005']
  columns[:2] : ['DV0IC8ZOBR', '2424PF32OZ']
  range       : [0.000, 4330.120]  median=4.930
  zeros       : 4.1%
  in common_genes_v4: 181/200


In [41]:
RAW = Path("data/raw")

def build_raw(coh, fname, ref_ids):
    e = pd.read_csv(RAW / fname, index_col=0)
    missing = [g for g in genes if g not in e.index]
    assert not missing, f"{coh}: {len(missing)} of the 14,675 genes absent from raw file"
    e = e.loc[genes]                                    # enforce gene order == b1/b2

    # sample-ID reconciliation against the v4 arm
    cols = list(e.columns)
    if len(set(cols) & set(ref_ids)) < len(ref_ids):
        alt = [c.replace(".", "-") for c in cols]
        if len(set(alt) & set(ref_ids)) > len(set(cols) & set(ref_ids)):
            e.columns = alt
            print(f"  {coh}: converted dots→dashes in sample IDs")
    hit = set(e.columns) & set(ref_ids)
    assert len(hit) == len(ref_ids), \
        f"{coh}: matched {len(hit)}/{len(ref_ids)} samples; ID formats differ"

    e = e[ref_ids]                                      # exact same patients, same order
    X = np.log2(e.values.T.astype(np.float64) + 1.0)    # stage 2
    X = (X - X.mean(0)) / X.std(0, ddof=1)              # stage 4: z-score per gene
    X = np.clip(X, -3.0, 3.0)                           # stage 5
    print(f"  {coh}: {X.shape}  range=[{X.min():.2f}, {X.max():.2f}]  "
          f"median|mean|={np.median(np.abs(X.mean(0))):.4f}")
    return X

# recover the exact v4 sample order used in the fits
_st = pd.read_csv(PROC / "surv_tcga_aligned_v4.csv")
_so = pd.read_csv(PROC / "surv_orien_aligned_v4.csv")
_kt = "sampleID" if "sampleID" in _st.columns else _st.columns[0]
_ko = "sampleID" if "sampleID" in _so.columns else _so.columns[0]
_et = pd.read_csv(PROC / "tcga_preprocessed_v4.csv",  index_col=0, nrows=0)
_eo = pd.read_csv(PROC / "orien_preprocessed_v4.csv", index_col=0, nrows=0)
ids_t = [s for s in _et.columns if s in set(_st[_kt])]
ids_o = [s for s in _eo.columns if s in set(_so[_ko])]
assert len(ids_t) == len(tt) and len(ids_o) == len(to), "sample count mismatch vs v4 arm"

print("--- raw arm (no ComBat) ---")
Xt_raw = build_raw("tcga",  "expr_tmp_tcga.csv",       ids_t)
Xo_raw = build_raw("orien", "expr_tmp_orien_1112.csv", ids_o)

--- raw arm (no ComBat) ---
  tcga: (339, 14675)  range=[-3.00, 3.00]  median|mean|=0.0013
  orien: (1112, 14675)  range=[-3.00, 3.00]  median|mean|=0.0026


In [42]:
def arm_stats(Xa, Xb, label):
    ba, sa = cox_one_step(Xa, tt, et)
    bb, sb = cox_one_step(Xb, to, eo)
    m1, m2 = float(np.mean(sa**2)), float(np.mean(sb**2))
    V1 = float(np.var(ba, ddof=1)) - m1
    V2 = float(np.var(bb, ddof=1)) - m2
    Sxy = float(np.cov(ba, bb, ddof=1)[0, 1])
    c   = Sxy / V1
    tau2 = V2 - c**2 * V1
    Q = (ba - bb)**2 / (sa**2 + sb**2)
    return {
        "label": label,
        "lambda": float(np.median(Q) / CHI2_1_MEDIAN),
        "tau2": tau2, "c_reg": c,
        "r_obs": float(np.corrcoef(ba, bb)[0, 1]),
        "r_true": Sxy / np.sqrt(V1 * V2) if V1 > 0 and V2 > 0 else np.nan,
        "V1": V1, "V2": V2, "mse1": m1, "mse2": m2,
        "sign_conc": float(np.mean(np.sign(ba) == np.sign(bb))),
        "tail95": float(np.mean(Q > stats.chi2.ppf(0.95, 1))),
        "_b": (ba, bb, sa, sb),
    }

v4  = arm_stats(Xt,     Xo,     "v4 (ComBat)")
raw = arm_stats(Xt_raw, Xo_raw, "raw (no ComBat)")

print(f"\n{'':22s} {'v4 (ComBat)':>14} {'raw (no ComBat)':>16} {'change':>10}")
for k, fmt in [("tau2","{:.6f}"), ("lambda","{:.4f}"), ("r_true","{:.4f}"),
               ("r_obs","{:.4f}"), ("c_reg","{:.4f}"), ("sign_conc","{:.4f}"),
               ("V1","{:.6f}"), ("V2","{:.6f}"), ("mse1","{:.6f}"), ("mse2","{:.6f}"),
               ("tail95","{:.4f}")]:
    a, b = v4[k], raw[k]
    ch = f"{100*(b-a)/a:+.1f}%" if a not in (0, None) and np.isfinite(a) else "—"
    print(f"  {k:20s} {fmt.format(a):>14} {fmt.format(b):>16} {ch:>10}")

# sanity: v4 one-step must reproduce the stored MLE values
print(f"\nv4 one-step vs stored MLE:  lambda {v4['lambda']:.4f} vs 1.4157   "
      f"tau2 {v4['tau2']:.6f} vs 0.002648   r_true {v4['r_true']:.4f} vs 0.6737")


                          v4 (ComBat)  raw (no ComBat)     change
  tau2                       0.002682         0.003620     +35.0%
  lambda                       1.4099           2.9562    +109.7%
  r_true                       0.6584           0.6048      -8.1%
  r_obs                        0.2312           0.2125      -8.1%
  c_reg                        1.1461           1.1905      +3.9%
  sign_conc                    0.5672           0.4943     -12.8%
  V1                         0.001562         0.001473      -5.7%
  V2                         0.004734         0.005708     +20.6%
  mse1                       0.007031         0.007032      +0.0%
  mse2                       0.002242         0.002304      +2.7%
  tail95                       0.0942           0.2145    +127.6%

v4 one-step vs stored MLE:  lambda 1.4099 vs 1.4157   tau2 0.002682 vs 0.002648   r_true 0.6584 vs 0.6737


In [43]:
B_NULL   = 200
RUN_RAW  = True                      # set False to halve runtime
rng_null = np.random.default_rng(2026)

def genome_stats(ba, sa, bb, sb):
    Q = (ba - bb)**2 / (sa**2 + sb**2)
    return {
        "lambda":    float(np.median(Q) / CHI2_1_MEDIAN),
        "tail95":    float(np.mean(Q > stats.chi2.ppf(0.95,  1))),
        "tail99":    float(np.mean(Q > stats.chi2.ppf(0.99,  1))),
        "tail999":   float(np.mean(Q > stats.chi2.ppf(0.999, 1))),
        "r_obs":     float(np.corrcoef(ba, bb)[0, 1]),
        "sign_conc": float(np.mean(np.sign(ba) == np.sign(bb))),
    }

# observed, one-step (comparable to the null by construction)
obs_v4  = genome_stats(*[v4["_b"][i]  for i in (0, 1, 2, 3)][:1] + list(v4["_b"][1:]))  \
          if False else genome_stats(v4["_b"][0],  v4["_b"][2],  v4["_b"][1],  v4["_b"][3])
obs_raw = genome_stats(raw["_b"][0], raw["_b"][2], raw["_b"][1], raw["_b"][3])

null_v4, null_raw = [], []
print(f"--- permutation null for lambda (B={B_NULL}) ---")
for b in range(B_NULL):
    it = rng_null.permutation(len(tt))      # permutes (time, event) PAIRS together
    io = rng_null.permutation(len(to))      # independent per cohort; never across cohorts
    pt, ps = cox_one_step(Xt, tt[it], et[it])
    po, qs = cox_one_step(Xo, to[io], eo[io])
    null_v4.append(genome_stats(pt, ps, po, qs))
    if RUN_RAW:
        pt2, ps2 = cox_one_step(Xt_raw, tt[it], et[it])
        po2, qs2 = cox_one_step(Xo_raw, to[io], eo[io])
        null_raw.append(genome_stats(pt2, ps2, po2, qs2))
    if (b + 1) % 25 == 0:
        print(f"  {b+1}/{B_NULL}")

--- permutation null for lambda (B=200) ---
  25/200
  50/200
  75/200
  100/200
  125/200
  150/200
  175/200
  200/200


In [44]:
STATS = ["lambda", "tail95", "tail99", "tail999", "r_obs", "sign_conc"]

def report(obs, nulls, label):
    print(f"\n=== {label} ===")
    print(f"  {'stat':10} {'observed':>10} {'null mean':>10} {'null SD':>9} "
          f"{'z':>8} {'emp p':>8}  {'null 2.5–97.5%':>20}")
    rows = []
    for s in STATS:
        v  = np.array([d[s] for d in nulls])
        o  = obs[s]
        z  = (o - v.mean()) / v.std(ddof=1)
        p  = (1 + (v >= o).sum()) / (len(v) + 1)
        lo, hi = np.percentile(v, [2.5, 97.5])
        print(f"  {s:10} {o:>10.4f} {v.mean():>10.4f} {v.std(ddof=1):>9.4f} "
              f"{z:>8.2f} {p:>8.4f}  [{lo:.4f}, {hi:.4f}]")
        rows.append({"stat": s, "observed": o, "null_mean": float(v.mean()),
                     "null_sd": float(v.std(ddof=1)), "z": float(z), "emp_p": float(p),
                     "null_p2.5": float(lo), "null_p97.5": float(hi)})
    return rows

res = {"v4": report(obs_v4, null_v4, "v4 (ComBat)")}
if RUN_RAW:
    res["raw"] = report(obs_raw, null_raw, "raw (no ComBat)")

with open(OUTDIR / "permutation_null.json", "w") as f:
    json.dump({"B": B_NULL, "n_genes": int(N), "estimator": "one-step score at beta=0",
               "results": res}, f, indent=2)
print(f"\nwritten: {OUTDIR / 'permutation_null.json'}")


=== v4 (ComBat) ===
  stat         observed  null mean   null SD        z    emp p        null 2.5–97.5%
  lambda         1.4099     1.0288    0.1857     2.05   0.0348  [0.8057, 1.4222]
  tail95         0.0942     0.0516    0.0175     2.43   0.0199  [0.0301, 0.0870]
  tail99         0.0259     0.0107    0.0061     2.50   0.0249  [0.0042, 0.0237]
  tail999        0.0025     0.0011    0.0011     1.22   0.0697  [0.0001, 0.0039]
  r_obs          0.2312    -0.0020    0.0791     2.95   0.0050  [-0.1665, 0.1475]
  sign_conc      0.5672     0.4995    0.0337     2.01   0.0199  [0.4291, 0.5590]

=== raw (no ComBat) ===
  stat         observed  null mean   null SD        z    emp p        null 2.5–97.5%
  lambda         2.9562     1.0476    0.2472     7.72   0.0050  [0.7977, 1.6495]
  tail95         0.2145     0.0526    0.0221     7.32   0.0050  [0.0279, 0.1143]
  tail99         0.0842     0.0109    0.0073    10.08   0.0050  [0.0036, 0.0322]
  tail999        0.0191     0.0012    0.0012    15.10 

In [46]:
IQR_FILE = PROC / "iqr_filtered_v4_prez" / "retained_genes.txt"
iqr_set  = {g.strip() for g in open(IQR_FILE) if g.strip()}
mask     = np.array([g in iqr_set for g in genes])
print(f"IQR list: {len(iqr_set)}  |  matched in current universe: {mask.sum()}")

def bh_fdr(p):
    p = np.asarray(p, float); m = len(p); o = np.argsort(p)
    q = np.empty(m); r = p[o] * m / np.arange(1, m + 1)
    q[o] = np.clip(np.minimum.accumulate(r[::-1])[::-1], 0, 1)
    return q

def summarize(sel, label):
    ba, sa, bb, sb = b1[sel], s1[sel], b2[sel], s2[sel]
    gn = np.array(genes)[sel]
    n  = len(ba)
    m1, m2 = float(np.mean(sa**2)), float(np.mean(sb**2))
    V1 = float(np.var(ba, ddof=1)) - m1
    V2 = float(np.var(bb, ddof=1)) - m2
    Sxy = float(np.cov(ba, bb, ddof=1)[0, 1])
    Q   = (ba - bb)**2 / (sa**2 + sb**2)
    Vb  = 0.5 * (V1 + V2)
    ok  = V1 > 0 and V2 > 0
    pa, pb = 2*stats.norm.sf(np.abs(ba/sa)), 2*stats.norm.sf(np.abs(bb/sb))
    w1, w2 = 1/sa**2, 1/sb**2
    bp = (w1*ba + w2*bb)/(w1+w2); pp = 2*stats.norm.sf(np.abs(bp*np.sqrt(w1+w2)))
    z1, z2 = np.abs(ba/sa), np.abs(bb/sb)
    topk = {}
    for k in (50, 100, 500, 1000):
        if k > n: continue
        st = set(gn[np.argsort(-z1)[:k]]); so = set(gn[np.argsort(-z2)[:k]])
        ov, ex = len(st & so), k*k/n
        topk[k] = (ov, ex, ov/ex, float(stats.hypergeom.sf(ov-1, n, k, k)))
    return {
        "label": label, "n": n, "lambda": float(np.median(Q)/CHI2_1_MEDIAN),
        "tail95": float(np.mean(Q > stats.chi2.ppf(0.95, 1))),
        "r_obs_p": float(np.corrcoef(ba, bb)[0,1]),
        "r_obs_s": float(stats.spearmanr(ba, bb).statistic),
        "r_ceiling": float(Vb/np.sqrt((Vb+m1)*(Vb+m2))) if ok else np.nan,
        "r_true": Sxy/np.sqrt(V1*V2) if ok else np.nan,
        "V1": V1, "V2": V2, "snr1": V1/m1, "snr2": V2/m2,
        "sign_conc": float(np.mean(np.sign(ba) == np.sign(bb))),
        "fdr_T": int((bh_fdr(pa) < .05).sum()), "fdr_O": int((bh_fdr(pb) < .05).sum()),
        "fdr_pool": int((bh_fdr(pp) < .05).sum()),
        "fdr_Q": int((bh_fdr(stats.chi2.sf(Q,1)) < .05).sum()),
        "topk": topk,
    }

full = summarize(np.ones(len(genes), bool), "14,675 (all common)")
iqr  = summarize(mask, f"{int(mask.sum())} (IQR)")

keys = ["n","lambda","tail95","r_obs_p","r_obs_s","r_ceiling","r_true",
        "V1","V2","snr1","snr2","sign_conc","fdr_T","fdr_O","fdr_pool","fdr_Q"]
print(f"\n{'':12} {'all common':>14} {'IQR':>14}")
for k in keys:
    a, b = full[k], iqr[k]
    f = "{:>14.4f}" if isinstance(a, float) else "{:>14d}"
    print(f"  {k:10} " + f.format(a) + f.format(b))

for r in (full, iqr):
    print(f"\n--- top-k overlap | {r['label']} ---")
    print(f"  {'k':>5} {'obs':>5} {'exp':>7} {'fold':>6} {'p_hyper':>10}")
    for k, (ov, ex, fold, p) in r["topk"].items():
        print(f"  {k:>5} {ov:>5} {ex:>7.1f} {fold:>6.2f} {p:>10.2e}")

IQR list: 4835  |  matched in current universe: 4835

                 all common            IQR
  n                   14675          4835
  lambda             1.4157        1.4247
  tail95             0.0951        0.0904
  r_obs_p            0.2408        0.2953
  r_obs_s            0.2303        0.2877
  r_ceiling          0.4304        0.4855
  r_true             0.6737        0.7994
  V1                 0.0016        0.0015
  V2                 0.0048        0.0063
  snr1               0.2307        0.2259
  snr2               2.1393        2.8514
  sign_conc          0.5669        0.5888
  fdr_T                   0             0
  fdr_O                1860           920
  fdr_pool             1643           879
  fdr_Q                   0             0

--- top-k overlap | 14,675 (all common) ---
      k   obs     exp   fold    p_hyper
     50     1     0.2   5.87   1.57e-01
    100     1     0.7   1.47   4.96e-01
    500    26    17.0   1.53   2.17e-02
   1000    76    68.1   1.

In [47]:
N_STAB, N_T, N_O, N_SHARED = int(mask.sum()), 12, 14, 1   # verify against your files
exp_ov = N_T * N_O / N_STAB
p_ov   = float(stats.hypergeom.sf(N_SHARED - 1, N_STAB, N_T, N_O))
print(f"stability selection in the {N_STAB}-gene universe:")
print(f"  TCGA-stable {N_T}, ORIEN-stable {N_O}, shared {N_SHARED}")
print(f"  expected under independence: {exp_ov:.4f}   fold: {N_SHARED/exp_ov:.1f}x   p = {p_ov:.4f}")

stability selection in the 4835-gene universe:
  TCGA-stable 12, ORIEN-stable 14, shared 1
  expected under independence: 0.0347   fold: 28.8x   p = 0.0342


In [49]:
for coh in ("tcga", "orien"):
    c = pd.read_csv(PROC / f"clinical_{coh}_aligned_v4.csv", index_col=0)
    print(f"\n=== {coh.upper()} === {c.shape}  columns: {list(c.columns)}")
    for v in ("smoking", "alcohol", "gender", "N_stage", "T_stage"):
        if v in c.columns:
            print(f"  {v:9} dtype={str(c[v].dtype):8} "
                  f"{dict(c[v].value_counts(dropna=False))}")
    print(f"  age       dtype={c['age'].dtype}  "
          f"min={c['age'].min():.1f} median={c['age'].median():.1f} "
          f"max={c['age'].max():.1f}  NaN={c['age'].isna().sum()}")


=== TCGA === (339, 7)  columns: ['sampleID', 'age', 'gender', 'smoking', 'alcohol', 'N_stage', 'T_stage']
  smoking   dtype=str      {'Ever': 240, 'Never': 91, 'Unknown': 8}
  alcohol   dtype=str      {'Ever': 221, 'Never': 110, 'Unknown': 8}
  gender    dtype=int64    {1: 233, 0: 106}
  N_stage   dtype=str      {'N0': 170, 'N2-3': 95, 'N1': 59, 'Unknown': 15}
  T_stage   dtype=str      {'T3-T4': 199, 'T1-T2': 129, 'Unknown': 11}
  age       dtype=int64  min=19.0 median=61.0 max=90.0  NaN=0

=== ORIEN === (1112, 7)  columns: ['sampleID', 'age', 'gender', 'smoking', 'alcohol', 'N_stage', 'T_stage']
  smoking   dtype=str      {'Ever': 718, 'Never': 340, 'Unknown': 54}
  alcohol   dtype=str      {'Ever': 667, 'Never': 337, 'Unknown': 108}
  gender    dtype=int64    {1: 760, 0: 352}
  N_stage   dtype=str      {'N0': 408, 'Unknown': 292, 'N2-3': 254, 'N1': 158}
  T_stage   dtype=str      {'Unknown': 413, 'T1-T2': 364, 'T3-T4': 335}
  age       dtype=float64  min=17.9 median=61.2 max=85.9  

In [50]:
COVARS = ["age", "gender", "smoking", "alcohol"]

def build_design(coh, ref_ids):
    c = pd.read_csv(PROC / f"clinical_{coh}_aligned_v4.csv", index_col=0)
    key = "sampleID" if "sampleID" in c.columns else c.index.name
    c = c.set_index("sampleID") if "sampleID" in c.columns else c
    c = c.loc[ref_ids]
    Z = pd.DataFrame(index=c.index)
    Z["age"]    = c["age"].astype(float)
    Z["gender"] = c["gender"].astype(float)
    for v in ("smoking", "alcohol"):                 # reference = 'Never'
        Z[f"{v}_Ever"]    = (c[v] == "Ever").astype(float)
        Z[f"{v}_Unknown"] = (c[v] == "Unknown").astype(float)
    return Z.values, list(Z.columns)

Zt, znames = build_design("tcga",  ids_t)
Zo, _      = build_design("orien", ids_o)
print(f"design: {znames}\n  TCGA {Zt.shape}  ORIEN {Zo.shape}")

def cox_adjusted_one_step(X, Z, time, event):
    """Score test for each gene added to the covariate-only Cox model.
    Fits the null model once (lifelines MLE), then residualizes genes against
    covariates in the risk-set metric. Returns (beta, se) per gene."""
    from lifelines import CoxPHFitter
    dfz = pd.DataFrame(Z, columns=znames); dfz["time"] = time; dfz["event"] = event
    cph = CoxPHFitter(penalizer=0.0).fit(dfz, "time", "event")
    gamma = cph.params_[znames].values
    w = np.exp(Z @ gamma)                                    # risk weights from null model

    n, p = X.shape; q = Z.shape[1]
    o = np.argsort(time, kind="mergesort")
    Xs, Zs, ws, ts, es = X[o], Z[o], w[o], time[o], event[o]

    S0 = 0.0
    S1x = np.zeros(p); S1z = np.zeros(q)
    S2xx = np.zeros(p); S2zz = np.zeros((q, q)); S2xz = np.zeros((p, q))
    U = np.zeros(p); Vxx = np.zeros(p); Vzz = np.zeros((q, q)); Vxz = np.zeros((p, q))

    i = n - 1
    while i >= 0:
        j = i
        while j > 0 and ts[j-1] == ts[i]:
            j -= 1
        bx, bz, bw = Xs[j:i+1], Zs[j:i+1], ws[j:i+1]
        S0   += bw.sum()
        S1x  += bw @ bx;              S1z  += bw @ bz
        S2xx += bw @ (bx**2);         S2zz += bz.T @ (bz * bw[:, None])
        S2xz += (bx * bw[:, None]).T @ bz
        ev = es[j:i+1] == 1; ne = int(ev.sum())
        if ne:
            mx, mz = S1x / S0, S1z / S0
            U   += bx[ev].sum(0) - ne * mx
            Vxx += ne * (S2xx / S0 - mx**2)
            Vzz += ne * (S2zz / S0 - np.outer(mz, mz))
            Vxz += ne * (S2xz / S0 - np.outer(mx, mz))
        i = j - 1

    Vzz_inv = np.linalg.pinv(Vzz)
    Vadj = np.maximum(Vxx - np.einsum("pq,qr,pr->p", Vxz, Vzz_inv, Vxz), 1e-12)
    return U / Vadj, 1.0 / np.sqrt(Vadj)

b1_m2, s1_m2 = cox_adjusted_one_step(Xt, Zt, tt, et)
b2_m2, s2_m2 = cox_adjusted_one_step(Xo, Zo, to, eo)

def gstats(ba, sa, bb, sb, label):
    m1, m2_ = float(np.mean(sa**2)), float(np.mean(sb**2))
    V1 = float(np.var(ba, ddof=1)) - m1
    V2 = float(np.var(bb, ddof=1)) - m2_
    Sxy = float(np.cov(ba, bb, ddof=1)[0, 1]); c = Sxy / V1
    Q = (ba - bb)**2 / (sa**2 + sb**2)
    return {"label": label, "lambda": float(np.median(Q)/CHI2_1_MEDIAN),
            "tau2": V2 - c**2 * V1, "c_reg": c,
            "r_obs": float(np.corrcoef(ba, bb)[0,1]),
            "r_true": Sxy/np.sqrt(V1*V2) if V1>0 and V2>0 else np.nan,
            "V1": V1, "V2": V2, "mse1": m1, "mse2": m2_,
            "sign_conc": float(np.mean(np.sign(ba)==np.sign(bb))),
            "tail95": float(np.mean(Q > stats.chi2.ppf(0.95,1)))}

m0 = gstats(b1_os, s1_os, b2_os, s2_os, "M0 (one-step)")
m2 = gstats(b1_m2, s1_m2, b2_m2, s2_m2, "M2 (one-step)")
print(f"\n{'':12} {'M0':>12} {'M2':>12} {'change':>10}")
for k in ("lambda","tau2","r_true","r_obs","sign_conc","c_reg","V1","V2","mse1","mse2","tail95"):
    a, b = m0[k], m2[k]
    print(f"  {k:10} {a:>12.6f} {b:>12.6f} {100*(b-a)/a:>9.1f}%")
print(f"\nbeta correlation M0 vs M2:  TCGA {np.corrcoef(b1_os,b1_m2)[0,1]:.4f}   "
      f"ORIEN {np.corrcoef(b2_os,b2_m2)[0,1]:.4f}")

design: ['age', 'gender', 'smoking_Ever', 'smoking_Unknown', 'alcohol_Ever', 'alcohol_Unknown']
  TCGA (339, 6)  ORIEN (1112, 6)

                       M0           M2     change
  lambda         1.409852     1.466600       4.0%
  tau2           0.002682     0.002705       0.9%
  r_true         0.658381     0.674566       2.5%
  r_obs          0.231227     0.249623       8.0%
  sign_conc      0.567155     0.566882      -0.0%
  c_reg          1.146118     1.112502      -2.9%
  V1             0.001562     0.001825      16.8%
  V2             0.004734     0.004963       4.8%
  mse1           0.007031     0.007296       3.8%
  mse2           0.002242     0.002288       2.0%
  tail95         0.094242     0.098739       4.8%

beta correlation M0 vs M2:  TCGA 0.9803   ORIEN 0.9804
